# M07b overnight: train D1 v2 -> pre-registered epoch choice -> freeze

**Unattended, one Run all on a T4.** Thin launcher over the same CLI as the separate
notebooks (`M07b_train_d1.ipynb`, `M07b_freeze_d1.ipynb`):

    python -m pii_redteam.cli train-d1  --config configs/m07b_train_d1_v2.yaml
    python -m pii_redteam.cli freeze-d1 --config configs/m07b_freeze_d1_v2.yaml --source-run-dir <run>

**Before Run all:** Runtime -> Change runtime type -> **T4 GPU**. Drive must hold frozen D0 at
`pii-redteam-rl/d0-20261004-142714/d0-best` and the AI4Privacy cache at `pii-redteam-rl/cache`.
The only prompt is the Drive mount in the first cell. Nothing else waits for input.

Stages (each one's status and wall-clock time go to
`pii-redteam-rl/d1v2-overnight/progress.json` as soon as it starts and ends):

1. **train**: D0 -> D1 with the v1 recipe, epochs 2 and 3 saved (`d1-epoch2/`, `d1-epoch3/`
   with sha256s). Skipped if a finished `d1v2-*` run on Drive already has both epochs with
   matching shas.
2. **choice**: the pre-registered dev-only rule (fewest name-free dev rows flagged, within
   0.01 of D0's dev F1; ties -> higher dev F1 -> earlier epoch). Read from the training report.
   freeze-d1 recomputes it and checks they agree.
3. **freeze**: locks, single-use final-eval for both epochs, short negatives (95 gated rows +
   16 homograph report-only), the M07a probe on the chosen epoch (timing projection printed,
   no wait), then `frozen` or `freeze_blocked`. Skipped if this run is already frozen.
4. **summary**: chosen epoch, gate table, status, paths, experiments_entry.

**Resuming after a disconnect:** Run all again. A finished training run is reused, final-eval
reports are reused (single use per candidate), a finished probe for the chosen sha is reused.
A training run or probe that was cut off mid-way is restarted, and a frozen dir is never
overwritten. A failed stage is recorded and the later cells still print the summary.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/GauthierRoy/pii-redteam-rl.git"
REPO_DIR = "/content/pii-redteam-rl"
DRIVE_ROOT = "/content/drive/MyDrive/pii-redteam-rl"
# Change to "main" once feat/m07b-freeze-d1 is merged.
REF = "feat/m07b-freeze-d1"

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Always sync to the target ref: a reused VM may hold a STALE clone.
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "fetch", "-q", "origin", REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "-q", "-B", REF, f"origin/{REF}"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REF}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", REF, REPO_URL, REPO_DIR], check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit:", commit)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True
)

import torch

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU."
free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb >= 10, (
    f"GPU has only {free_gb:.1f} GB free - dirty runtime. Runtime -> Disconnect and"
    " delete runtime, then reopen from GitHub and Run all."
)
print(f"GPU free: {free_gb:.1f} GB - OK ({torch.cuda.get_device_name(0)})")
ENV = dict(os.environ, PYTHONPATH=os.path.join(REPO_DIR, "src"), PROBE_REPO_COMMIT=commit)

In [ ]:
import datetime
import glob
import json
import time

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from pii_redteam.overnight import ProgressLog, find_reusable_train_run

TRAIN_CONFIG = "configs/m07b_train_d1_v2.yaml"
FREEZE_CONFIG = "configs/m07b_freeze_d1_v2.yaml"
SAVE_EPOCHS = [2, 3]  # must match training.save_epochs in TRAIN_CONFIG
OVERNIGHT_DIR = os.path.join(DRIVE_ROOT, "d1v2-overnight")
os.makedirs(OVERNIGHT_DIR, exist_ok=True)
PROGRESS = ProgressLog(
    os.path.join(OVERNIGHT_DIR, "progress.json"),
    session=f"{datetime.datetime.now(datetime.UTC):%Y%m%d-%H%M%S} commit {commit}",
)
STATE = {"train_run": None, "chosen_epoch": None, "freeze_report": None, "errors": []}


def run_cli(args, log_name):
    """Run one CLI step, streaming its output to the cell AND to a log file on Drive."""
    log_path = os.path.join(OVERNIGHT_DIR, log_name)
    print("$ python -m pii_redteam.cli", " ".join(args), f"\n  (log: {log_path})")
    with open(log_path, "a", encoding="utf-8") as log:
        log.write(f"\n===== {datetime.datetime.now(datetime.UTC).isoformat()} {args}\n")
        proc = subprocess.Popen(
            [sys.executable, "-u", "-m", "pii_redteam.cli", *args],
            cwd=REPO_DIR, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        return proc.wait()


print("progress file:", PROGRESS.path)

## Stage 1: train D1 v2 (epochs 2 and 3 saved)

In [ ]:
# ---- stage 1: train (or reuse a finished run)
PROGRESS.start("train", config=TRAIN_CONFIG)
try:
    reuse, seen = find_reusable_train_run(DRIVE_ROOT, "d1v2", SAVE_EPOCHS)
    for s in seen:
        print(f"  found {os.path.basename(s['run_dir'])}: {s['reason']}")
    if reuse:
        STATE["train_run"] = reuse
        print("REUSING finished training run:", reuse)
        PROGRESS.end("train", "reused", run_dir=reuse)
    else:
        before = set(glob.glob(os.path.join(DRIVE_ROOT, "d1v2-2*")))
        code = run_cli(["train-d1", "--config", TRAIN_CONFIG], "train.log")
        new_runs = sorted(set(glob.glob(os.path.join(DRIVE_ROOT, "d1v2-2*"))) - before)
        run_dir = new_runs[-1] if new_runs else None
        reuse, seen = find_reusable_train_run(DRIVE_ROOT, "d1v2", SAVE_EPOCHS)
        if code == 0 and run_dir and reuse == run_dir:
            STATE["train_run"] = run_dir
            PROGRESS.end("train", "done", run_dir=run_dir, exit_code=code)
        else:
            why = next((s["reason"] for s in seen if s["run_dir"] == run_dir), "no run dir")
            msg = f"training did not produce a usable run (exit {code}, {run_dir}: {why})"
            STATE["errors"].append(msg)
            PROGRESS.end("train", "failed", run_dir=run_dir, exit_code=code, error=msg)
            print("FAILED:", msg, "- the traceback above is the real error.")
except Exception as exc:  # recorded, so the summary cell still runs
    STATE["errors"].append(f"train: {exc!r}")
    PROGRESS.end("train", "failed", error=repr(exc))
    print("FAILED: train stage raised", repr(exc))

## Stage 2: pre-registered epoch choice (dev only)

In [ ]:
# ---- stage 2: the pre-registered, dev-only epoch choice (read from the training report)
PROGRESS.start("choice")
if STATE["train_run"]:
    with open(os.path.join(STATE["train_run"], "d1_report.json"), encoding="utf-8") as f:
        train_report = json.load(f)
    sel = train_report["freeze_selection"]
    STATE["chosen_epoch"] = sel["chosen_epoch"]
    print(f"rule {sel['rule']}: D0 dev F1 {sel['d0_dev_f1']}, floor -{sel['dev_f1_max_drop']}")
    for c in sel["candidates"]:
        mark = "CHOSEN" if c["epoch"] == sel["chosen_epoch"] else "      "
        print(f"  {mark} epoch {c['epoch']}: dev F1 {c['dev_span_f1']}, name-free dev rows "
              f"flagged {c['dev_neg_rows_with_pred']}")
    repro = train_report.get("reproduction")
    if repro:
        print("reproduction of the v1 epoch 2:", repro["note"])
    PROGRESS.end("choice", "done" if sel["chosen_epoch"] else "failed",
                 chosen_epoch=sel["chosen_epoch"], eligible=sel["eligible_epochs"],
                 reproduction=repro)
else:
    PROGRESS.end("choice", "skipped", reason="no training run")
    print("SKIPPED: no usable training run.")

## Stage 3: freeze

In [ ]:
# ---- stage 3: freeze (final-eval both epochs, short negatives, probe on the chosen one)
PROGRESS.start("freeze", config=FREEZE_CONFIG)
try:
    if not STATE["train_run"]:
        PROGRESS.end("freeze", "skipped", reason="no training run")
        print("SKIPPED: no usable training run.")
    else:
        run_id = os.path.basename(STATE["train_run"])
        frozen = os.path.join(DRIVE_ROOT, f"d1-frozen-{run_id}")
        earlier = sorted(glob.glob(os.path.join(DRIVE_ROOT, "d1v2-freeze-2*", "d1_freeze_report.json")))
        if os.path.exists(os.path.join(frozen, "freeze_manifest.json")):
            # Already frozen in an earlier session: never overwrite, just report it.
            for path in reversed(earlier):
                with open(path, encoding="utf-8") as f:
                    rep = json.load(f)
                if rep.get("status") == "frozen" and rep.get("frozen_dir") == frozen:
                    STATE["freeze_report"] = path
                    break
            print("ALREADY FROZEN:", frozen)
            PROGRESS.end("freeze", "already_frozen", frozen_dir=frozen,
                         report=STATE["freeze_report"])
        else:
            before = set(glob.glob(os.path.join(DRIVE_ROOT, "d1v2-freeze-2*")))
            code = run_cli(
                ["freeze-d1", "--config", FREEZE_CONFIG, "--source-run-dir", STATE["train_run"]],
                "freeze.log",
            )
            new = sorted(set(glob.glob(os.path.join(DRIVE_ROOT, "d1v2-freeze-2*"))) - before)
            path = os.path.join(new[-1], "d1_freeze_report.json") if new else None
            if path and os.path.exists(path):
                STATE["freeze_report"] = path
                with open(path, encoding="utf-8") as f:
                    status = json.load(f)["status"]
            else:
                status = "failed"
            if code not in (0, 2):
                STATE["errors"].append(f"freeze-d1 exited {code}; see freeze.log")
                status = "failed" if status == "running" else status
            PROGRESS.end("freeze", status, exit_code=code, report=path)
            print(f"freeze-d1 exit code {code} -> status {status}")
except Exception as exc:
    STATE["errors"].append(f"freeze: {exc!r}")
    PROGRESS.end("freeze", "failed", error=repr(exc))
    print("FAILED: freeze stage raised", repr(exc))

## Summary

In [ ]:
def print_freeze_summary(report_path):
    with open(report_path, encoding="utf-8") as f:
        report = json.load(f)
    print("STATUS:", report["status"])
    print("chosen candidate:", report.get("chosen_candidate"),
          "| rule:", (report.get("selection") or {}).get("rule"))
    for row in report.get("side_by_side") or []:
        mark = "CHOSEN" if row["chosen"] else "      "
        print(f"  {mark} {row['name']} (epoch {row['epoch']}): final-eval F1 "
              f"{row['final_eval_f1']} P {row['final_eval_precision']} R "
              f"{row['final_eval_recall']} name-free rows flagged "
              f"{row['final_eval_neg_rows_with_pred']} | short-neg gated rate "
              f"{row['short_negatives_gated_fp_row_rate']} | probe ok {row['probe_ok']}")
        print(f"         gates {row['gates_ok']}")
    print("checks:")
    for name, c in report["checks"].items():
        req = "" if c["required"] else " (optional)"
        print(f"  [{c['status'].upper()}] {name}{req}")
    if report["status"] == "frozen":
        print("frozen dir:", report.get("frozen_dir"))
    print("report:", report_path)
    print("\nexperiments_entry (paste into EXPERIMENTS.yaml):")
    print(json.dumps(report.get("experiments_entry"), indent=2))
    return report


print("=" * 78)
print("OVERNIGHT SUMMARY")
print("=" * 78)
print("repo commit:", commit)
print("training run:", STATE["train_run"])
print("chosen epoch (pre-registered dev rule):", STATE["chosen_epoch"])
if STATE["freeze_report"]:
    final = print_freeze_summary(STATE["freeze_report"])
    PROGRESS.state["final_status"] = final["status"]
else:
    print("no freeze report.")
    PROGRESS.state["final_status"] = "incomplete"
print("\nstage timings (s):")
for stage, info in PROGRESS.session["stages"].items():
    print(f"  {stage:8s} {info.get('status'):15s} {info.get('seconds')}")
if STATE["errors"]:
    print("\nERRORS:")
    for e in STATE["errors"]:
        print("  -", e)
print("\nprogress:", PROGRESS.path)
print("logs:", OVERNIGHT_DIR)
PROGRESS.write()

**In the morning:** bring back `d1v2-overnight/progress.json`, the training run's
`d1_report.json`, and the freeze run's `d1_freeze_report.json` (its `experiments_entry` goes
into `EXPERIMENTS.yaml`, and so does the training run's). `freeze_blocked` is a recorded
result: do not edit a threshold or the epoch choice to get a pass without a recorded owner
decision.